In [7]:
import pandas as pd
import numpy as np
import glob
import os
import re
from sklearn.preprocessing import MinMaxScaler
import joblib  # To save the scaler for later use

BASE_PATH = "dataset/CSV Files"
WINDOW_SIZE = 128   # 128 samples (~0.12s) at 1000Hz
STRIDE = 64         # Overlap windows by 50% to get more training data
TRAIN_SPLIT = 0.8   # First 80% of Healthy data = Train

# Create output folder for processed data
if not os.path.exists("processed_data"):
    os.makedirs("processed_data")

In [8]:
def natural_sort_key(s):
    """
    Sorts strings containing numbers naturally (e.g. 'File(2)' before 'File(10)')
    """
    return [int(text) if text.isdigit() else text.lower()
            for text in re.split('([0-9]+)', s)]


def load_files_from_folder(folder_path):
    """
    Loads all CSVs in a folder, sorts them chronologically,
    and returns a single continuous signal (Vector Sum).
    """
    files = sorted(glob.glob(os.path.join(folder_path, "*.csv")),
                   key=natural_sort_key)

    combined_signal = []
    print(
        f"Loading {len(files)} files from {os.path.basename(folder_path)}...")

    for f in files:
        try:
            # Read CSV (No header)
            df = pd.read_csv(f, header=None)

            # Vector Sum (Magnitude of Vibration)
            # sqrt(x^2 + y^2 + z^2)
            if df.shape[1] >= 3:
                data = df.iloc[:, :3].values
                magnitude = np.sqrt(np.sum(data**2, axis=1))
            else:
                magnitude = df.iloc[:, 0].values

            combined_signal.extend(magnitude)
        except Exception as e:
            print(f"Skipping corrupt file {f}: {e}")

    return np.array(combined_signal)


def create_windows(signal, window_size, stride):
    """
    Slices a continuous signal into fixed-size windows.
    Returns shape: (Num_Windows, Window_Size, 1)
    """
    num_windows = (len(signal) - window_size) // stride + 1
    windows = []

    for i in range(num_windows):
        start = i * stride
        end = start + window_size
        windows.append(signal[start:end])

    return np.array(windows).reshape(-1, window_size, 1)

In [9]:
# --- PROCESS HEALTHY DATA (Train & Test) ---
print("--- Processing Normal (Healthy) Data ---")
normal_signal = load_files_from_folder(os.path.join(BASE_PATH, "Normal"))

# Calculate Split Point (Chronological)
split_idx = int(len(normal_signal) * TRAIN_SPLIT)

train_signal_raw = normal_signal[:split_idx]  # First 80%
test_normal_raw = normal_signal[split_idx:]   # Last 20%

print(f"Total Healthy Samples: {len(normal_signal)}")
print(f"Training Raw Size: {len(train_signal_raw)}")
print(f"Test (Normal) Raw Size: {len(test_normal_raw)}")

# --- PROCESS FAULTY DATA (Test Only) ---
print("--- Processing Faulty Data ---")
inner_signal = load_files_from_folder(
    os.path.join(BASE_PATH, "Inner Race Fault"))
outer_signal = load_files_from_folder(
    os.path.join(BASE_PATH, "Outer Race Fault"))

# Combine all faulty data into one "Anomalies" array for testing
test_faulty_raw = np.concatenate([inner_signal, outer_signal])
print(f"Test (Faulty) Raw Size: {len(test_faulty_raw)}")

--- Processing Normal (Healthy) Data ---
Loading 2160 files from Normal...
Total Healthy Samples: 21600000
Training Raw Size: 17280000
Test (Normal) Raw Size: 4320000
--- Processing Faulty Data ---
Loading 2160 files from Inner Race Fault...
Loading 2160 files from Outer Race Fault...
Test (Faulty) Raw Size: 43200000


In [10]:
# Create Windows
X_train = create_windows(train_signal_raw, WINDOW_SIZE, STRIDE)
X_test_normal = create_windows(test_normal_raw, WINDOW_SIZE, STRIDE)
X_test_faulty = create_windows(test_faulty_raw, WINDOW_SIZE, STRIDE)

print("--- Shape Check ---")
print(f"X_train shape: {X_train.shape}")
print(f"X_test_normal shape: {X_test_normal.shape}")
print(f"X_test_faulty shape: {X_test_faulty.shape}")

--- Shape Check ---
X_train shape: (269999, 128, 1)
X_test_normal shape: (67499, 128, 1)
X_test_faulty shape: (674999, 128, 1)


In [11]:
# Initialize Scaler
scaler = MinMaxScaler(feature_range=(0, 1))

# Flatten 3D -> 2D for scaling (Samples, Window_Size)
# We treat every window as a row of features temporarily
X_train_flat = X_train.reshape(-1, WINDOW_SIZE)
X_test_normal_flat = X_test_normal.reshape(-1, WINDOW_SIZE)
X_test_faulty_flat = X_test_faulty.reshape(-1, WINDOW_SIZE)

# FIT on Train, TRANSFORM everything
scaler.fit(X_train_flat)

X_train_scaled = scaler.transform(X_train_flat)
X_test_normal_scaled = scaler.transform(X_test_normal_flat)
X_test_faulty_scaled = scaler.transform(X_test_faulty_flat)

# Reshape back to 3D for Conv1D input: (Samples, Window_Size, 1)
X_train_final = X_train_scaled.reshape(-1, WINDOW_SIZE, 1)
X_test_normal_final = X_test_normal_scaled.reshape(-1, WINDOW_SIZE, 1)
X_test_faulty_final = X_test_faulty_scaled.reshape(-1, WINDOW_SIZE, 1)

print("Normalization Complete.")
print(f"Max value in Train: {np.max(X_train_final)}")
print(
    f"Max value in Faulty Test (Should be > 1.0 likely): {np.max(X_test_faulty_final)}")

Normalization Complete.
Max value in Train: 1.0000000000000002
Max value in Faulty Test (Should be > 1.0 likely): 1.3807382650019306


In [12]:
# Save Data Arrays
np.save("processed_data/X_train.npy", X_train_final)
np.save("processed_data/X_test_normal.npy", X_test_normal_final)
np.save("processed_data/X_test_faulty.npy", X_test_faulty_final)

# Save the Scaler (Important! We need this for the Arduino deployment later)
joblib.dump(scaler, "processed_data/scaler.save")

print("Files saved to 'processed_data/'")

Files saved to 'processed_data/'
